<a href="https://colab.research.google.com/github/Gabriel-Bela/Colonia-Inteligente/blob/main/Col%C3%B4nia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""Colônia Inteligente - sistema integrado em um único arquivo.

Uso:  python colonia.py

Módulos:
  1. Dados          -> hierarquia da colônia (dicionários aninhados) e listas
  2. Energia        -> geração x consumo x reserva
  3. Previsão       -> regressão linear simples (vento -> energia eólica)
  4. Prioridades    -> quem fica ligado e quem é desligado
  5. Modos          -> normal / reduzido / economia / emergência
  6. Decisões       -> regras com condições combinadas e prioridade
  7. Ciclo          -> junta tudo em uma simulação
  8. Menu           -> interface por terminal (todos os dados via input)
"""


# =============================================================================
# Entrada e saída (centralizadas para facilitar manutenção)
# =============================================================================

def pedir(texto):
    return input(texto)


def exibir(texto=""):
    print(texto)


def ler_numero(texto, minimo=None):
    """Lê um número (aceita vírgula). Repete até ser válido."""
    while True:
        bruto = pedir(texto).strip().replace(",", ".")
        try:
            valor = float(bruto)
        except ValueError:
            exibir("Valor inválido. Digite um número.")
            continue
        if minimo is not None and valor < minimo:
            exibir(f"O valor deve ser maior ou igual a {minimo:g}.")
            continue
        return valor


def ler_lista(texto):
    """Lê números separados por espaço. Ex.: 8 10 12"""
    while True:
        partes = pedir(texto).replace(",", ".").split()
        try:
            return [float(p) for p in partes]
        except ValueError:
            exibir("Lista inválida. Use apenas números separados por espaço.")


def formatar(valor):
    """Mostra 45.0 como 45 e 27.5 como 27.5."""
    return f"{valor:g}" if isinstance(valor, float) else str(valor)


# =============================================================================
# 1. DADOS - hierarquia da colônia
# =============================================================================

HISTORICO_VENTO = [8, 10, 12]
HISTORICO_ENERGIA_EOLICA = [20, 25, 30]
CAPACIDADE_RESERVA = 100


def criar_colonia():
    """Árvore de sistemas. Prioridade 1 = mais importante."""
    return {
        "energia": {
            "solar": {"geracao": 45, "status": "ativo"},
            "eolico": {"geracao": 25, "status": "ativo"},
            "reserva": 30,
        },
        "sistemas": {
            "suporte_vida": {"consumo": 30, "prioridade": 1, "essencial": True, "ligado": True},
            "habitacao": {"consumo": 20, "prioridade": 2, "essencial": True, "ligado": True},
            "comunicacao": {"consumo": 10, "prioridade": 3, "essencial": False, "ligado": True},
            "laboratorio": {"consumo": 15, "prioridade": 4, "essencial": False, "ligado": True},
            "entretenimento": {"consumo": 10, "prioridade": 5, "essencial": False, "ligado": True},
        },
        "clima": {"vento": 11, "temperatura": -20},
    }


def navegar(colonia, caminho):
    """Acessa um subsistema por caminho. Ex.: 'energia/solar/geracao'."""
    no = colonia
    for parte in caminho.strip("/").split("/"):
        if not isinstance(no, dict) or parte not in no:
            raise KeyError(f"Caminho inexistente: {caminho}")
        no = no[parte]
    return no


def total_geracao(colonia):
    """Soma a geração das fontes com status 'ativo'."""
    e = colonia["energia"]
    return sum(e[f]["geracao"] for f in ("solar", "eolico") if e[f]["status"] == "ativo")


def total_consumo(colonia):
    """Consumo atual: soma apenas dos sistemas ligados."""
    return sum(s["consumo"] for s in colonia["sistemas"].values() if s["ligado"])


def demanda_total(colonia):
    """Consumo que a colônia teria com todos os sistemas ligados."""
    return sum(s["consumo"] for s in colonia["sistemas"].values())


# =============================================================================
# 2. ANÁLISE DE ENERGIA
# =============================================================================

def analisar_energia(geracao, consumo, reserva=0):
    """Compara geração, consumo e reserva e devolve uma mensagem."""
    if consumo > geracao:
        if reserva >= consumo - geracao:
            return "ATENÇÃO: consumo maior que geração (coberto pela reserva)"
        return "ALERTA: consumo maior que geração"
    if geracao > consumo:
        return "SUGESTÃO: armazenar energia excedente"
    return "OK: geração e consumo equilibrados"


# =============================================================================
# 3. PREVISÃO - regressão linear simples (y = a*x + b)
# =============================================================================

def ajustar_reta(x, y):
    """Mínimos quadrados. Retorna (a, b)."""
    if len(x) != len(y) or len(x) < 2:
        raise ValueError("as listas devem ter o mesmo tamanho (mínimo 2 pontos).")
    n = len(x)
    media_x, media_y = sum(x) / n, sum(y) / n
    den = sum((xi - media_x) ** 2 for xi in x)
    if den == 0:
        raise ValueError("todos os valores de vento são iguais: não há reta.")
    a = sum((xi - media_x) * (yi - media_y) for xi, yi in zip(x, y)) / den
    return a, media_y - a * media_x


def prever(x, a, b):
    return a * x + b


def prever_energia_eolica(vento, hist_vento=None, hist_energia=None):
    """Prevê energia eólica; usa o histórico padrão se nenhum for informado."""
    hist_vento = HISTORICO_VENTO if hist_vento is None else hist_vento
    hist_energia = HISTORICO_ENERGIA_EOLICA if hist_energia is None else hist_energia
    a, b = ajustar_reta(hist_vento, hist_energia)
    return max(0.0, prever(vento, a, b))


# =============================================================================
# 4. PRIORIDADES - suporte à vida sempre ligado, não essenciais por último
# =============================================================================

def ordenar_por_prioridade(colonia):
    """Lista [(nome, dados)] da prioridade 1 (mais importante) em diante."""
    return sorted(colonia["sistemas"].items(), key=lambda item: (item[1]["prioridade"], item[0]))


def aplicar_prioridades(colonia, energia_disponivel, limite_prioridade=None):
    """Liga os sistemas em ordem de prioridade até a energia acabar.

    - Essenciais (suporte à vida, habitação) ficam SEMPRE ligados.
    - Quando um não essencial não cabe, ele e todos os de prioridade menor
      são desligados (nunca fica ligado algo menos importante que um desligado).
    - limite_prioridade (opcional): não essenciais com prioridade acima dele
      ficam desligados, mesmo que haja energia (limite do modo de operação).
    """
    saldo = energia_disponivel
    faltou = False
    ligados, desligados = [], []
    for nome, s in ordenar_por_prioridade(colonia):
        if s["essencial"]:
            s["ligado"] = True
        elif limite_prioridade is not None and s["prioridade"] > limite_prioridade:
            s["ligado"] = False
        elif not faltou and s["consumo"] <= saldo:
            s["ligado"] = True
        else:
            faltou = True
            s["ligado"] = False
        if s["ligado"]:
            saldo -= s["consumo"]
            ligados.append(nome)
        else:
            desligados.append(nome)
    return {
        "ligados": ligados,
        "desligados": desligados,
        "saldo": saldo,
        "essenciais_em_risco": saldo < 0,
    }


def desligar_nao_essenciais(colonia):
    """Desliga todos os sistemas não essenciais. Retorna os desligados."""
    desligados = []
    for nome, s in ordenar_por_prioridade(colonia):
        if not s["essencial"]:
            s["ligado"] = False
            desligados.append(nome)
    return desligados


# =============================================================================
# 5. MODOS DE OPERAÇÃO - até qual prioridade os sistemas permanecem ligados
# =============================================================================

MODOS = {
    "normal": 5,      # tudo ligado
    "reduzido": 4,    # desliga prioridade 5
    "economia": 3,    # desliga prioridades 4 e 5
    "emergencia": 2,  # mantém só suporte à vida e habitação
}


def aplicar_modo(colonia, modo):
    """Aplica um modo e devolve a lista dos sistemas desligados."""
    limite = MODOS[modo]
    desligados = []
    for nome, s in ordenar_por_prioridade(colonia):
        s["ligado"] = s["essencial"] or s["prioridade"] <= limite
        if not s["ligado"]:
            desligados.append(nome)
    return desligados


# =============================================================================
# 6. DECISÕES - regras com condições combinadas (1 = prioridade máxima)
# =============================================================================

LIM_CRITICO = 20
LIM_BAIXA = 50
LIM_CONSUMO_ALTO = 80

# (prioridade, ação, modo, condição(energia, consumo))
REGRAS = [
    (1, "CRÍTICO: manter só suporte à vida e desligar sistemas não essenciais",
     "emergencia", lambda e, c: e < LIM_CRITICO),
    (2, "ALERTA: ativar modo economia",
     "economia", lambda e, c: e < LIM_BAIXA and c >= LIM_CONSUMO_ALTO),
    (3, "ALERTA: reduzir consumo",
     "reduzido", lambda e, c: e < LIM_BAIXA),
    (4, "AVISO: consumo alto, monitorar",
     "normal", lambda e, c: c >= LIM_CONSUMO_ALTO),
]


def decidir_com_modo(energia, consumo):
    """Retorna (mensagem, modo) da regra de maior prioridade que se aplica."""
    for _, acao, modo, condicao in sorted(REGRAS, key=lambda r: r[0]):
        if condicao(energia, consumo):
            return acao, modo
    return "OK: operação normal", "normal"


def decidir(energia, consumo):
    return decidir_com_modo(energia, consumo)[0]


# =============================================================================
# 7. CICLO COMPLETO - previsão -> análise -> decisão -> prioridades -> reserva
# =============================================================================

def simular_ciclo(colonia, vento, saida=None):
    """Executa um ciclo e devolve o relatório (lista de linhas)."""
    linhas = []
    registrar = linhas.append

    colonia["clima"]["vento"] = vento
    previsao = prever_energia_eolica(vento)
    colonia["energia"]["eolico"]["geracao"] = round(previsao, 1)
    registrar(f"1) Previsão: vento {formatar(vento)} -> eólica ≈ {previsao:.1f}")

    geracao = total_geracao(colonia)
    reserva = colonia["energia"]["reserva"]
    demanda = demanda_total(colonia)
    registrar(f"2) Geração: {formatar(geracao)} | Demanda: {formatar(demanda)} | Reserva: {formatar(reserva)}")
    registrar("   " + analisar_energia(geracao, demanda, reserva))

    acao, modo = decidir_com_modo(geracao + reserva, demanda)
    registrar(f"3) Decisão: {acao} (modo {modo})")

    resultado = aplicar_prioridades(colonia, geracao + reserva, MODOS[modo])
    consumo = total_consumo(colonia)
    registrar("4) Ligados: " + ", ".join(resultado["ligados"]))
    desligados = resultado["desligados"]
    registrar("   Desligados: " + (", ".join(desligados) if desligados else "nenhum"))
    if resultado["essenciais_em_risco"]:
        registrar("   !!! ENERGIA INSUFICIENTE até para os sistemas essenciais !!!")

    nova_reserva = reserva + geracao - consumo
    nova_reserva = max(0, min(CAPACIDADE_RESERVA, nova_reserva))
    colonia["energia"]["reserva"] = nova_reserva
    registrar(f"5) Consumo final: {formatar(consumo)} | Reserva: {formatar(reserva)} -> {formatar(nova_reserva)}")
    return linhas


# =============================================================================
# 8. MENU INTERATIVO
# =============================================================================

def mostrar_painel(colonia):
    geracao = total_geracao(colonia)
    consumo = total_consumo(colonia)
    reserva = colonia["energia"]["reserva"]
    exibir("=== PAINEL DA COLÔNIA ===")
    exibir(f"Solar: {formatar(navegar(colonia, 'energia/solar/geracao'))} | "
           f"Eólico: {formatar(navegar(colonia, 'energia/eolico/geracao'))}")
    exibir(f"Geração: {formatar(geracao)} | Consumo: {formatar(consumo)} | Reserva: {formatar(reserva)}")
    exibir(f"Vento: {formatar(navegar(colonia, 'clima/vento'))} | "
           f"Temperatura: {formatar(navegar(colonia, 'clima/temperatura'))}")
    exibir(analisar_energia(geracao, consumo, reserva))


def mostrar_sistemas(colonia):
    exibir("Prioridade | Sistema          | Consumo | Estado")
    for nome, s in ordenar_por_prioridade(colonia):
        estado = "LIGADO" if s["ligado"] else "DESLIGADO"
        tipo = " (essencial)" if s["essencial"] else ""
        exibir(f"    P{s['prioridade']}     | {nome:<16} | {formatar(s['consumo']):>7} | {estado},{tipo}")


def opcao_navegar(colonia):
    caminho = pedir("Caminho (ex.: energia/solar/geracao ou sistemas/suporte_vida): ")
    try:
        exibir(f"{caminho} = {navegar(colonia, caminho)}")
    except KeyError as erro:
        exibir(f"Erro: {erro.args[0]}")


def opcao_analisar():
    geracao = ler_numero("Geração: ", 0)
    consumo = ler_numero("Consumo: ", 0)
    reserva = ler_numero("Reserva (0 se não houver): ", 0)
    exibir(analisar_energia(geracao, consumo, reserva))


def opcao_prever():
    usar_padrao = pedir("Usar histórico padrão (vento 8 10 12 / energia 20 25 30)? [s/n]: ").strip().lower()
    hist_vento = hist_energia = None
    if usar_padrao.startswith("n"):
        hist_vento = ler_lista("Histórico de vento (separe por espaço): ")
        hist_energia = ler_lista("Histórico de energia eólica (separe por espaço): ")
    vento = ler_numero("Velocidade do vento para prever: ", 0)
    try:
        previsao = prever_energia_eolica(vento, hist_vento, hist_energia)
    except ValueError as erro:
        exibir(f"Erro na regressão: {erro}")
        return
    exibir(f"Previsão de energia eólica ≈ {previsao:.1f}")


def opcao_decidir():
    energia = ler_numero("Energia: ", 0)
    consumo = ler_numero("Consumo: ", 0)
    acao, modo = decidir_com_modo(energia, consumo)
    exibir(acao)
    exibir(f"Modo de operação sugerido: {modo}")


def opcao_prioridades(colonia):
    exibir("Sistemas por prioridade (P1 = mais importante):")
    mostrar_sistemas(colonia)
    energia = ler_numero("\nEnergia disponível para distribuir: ", 0)
    resultado = aplicar_prioridades(colonia, energia)
    exibir("\nResultado da distribuição:")
    mostrar_sistemas(colonia)
    exibir(f"Energia restante: {formatar(resultado['saldo'])}")
    if resultado["essenciais_em_risco"]:
        exibir("ALERTA: energia insuficiente até para os sistemas essenciais!")
    elif resultado["desligados"]:
        exibir("Desligados por falta de energia: " + ", ".join(resultado["desligados"]))
    else:
        exibir("Todos os sistemas foram mantidos ligados.")


def opcao_modo(colonia):
    exibir("Modos: " + ", ".join(f"{m} (até P{p})" for m, p in MODOS.items()))
    modo = pedir("Modo desejado: ").strip().lower()
    if modo not in MODOS:
        exibir("Modo inválido.")
        return
    desligados = aplicar_modo(colonia, modo)
    exibir(f"Modo {modo} aplicado. Desligados: " + (", ".join(desligados) if desligados else "nenhum"))
    mostrar_sistemas(colonia)


def opcao_ciclo(colonia):
    vento = ler_numero("Velocidade do vento atual: ", 0)
    for linha in simular_ciclo(colonia, vento):
        exibir(linha)


def opcao_alterar(colonia):
    caminho = pedir("Caminho numérico a alterar (ex.: energia/reserva): ").strip()
    try:
        navegar(colonia, caminho)
    except KeyError as erro:
        exibir(f"Erro: {erro.args[0]}")
        return
    pai, _, chave = caminho.rpartition("/")
    destino = navegar(colonia, pai) if pai else colonia
    atual = destino[chave]
    if isinstance(atual, bool) or not isinstance(atual, (int, float)):
        exibir("Só é possível alterar valores numéricos.")
        return
    destino[chave] = ler_numero(f"Novo valor (atual {formatar(atual)}): ")
    exibir(f"{caminho} atualizado para {formatar(destino[chave])}")


MENU = """
===== COLÔNIA INTELIGENTE =====
1 - Ver painel da colônia
2 - Navegar pelos subsistemas
3 - Analisar uso de energia
4 - Prever energia eólica (regressão)
5 - Decidir ação (energia, consumo)
6 - Distribuir energia por prioridade
7 - Definir modo de operação
8 - Simular ciclo completo
9 - Alterar um dado numérico
0 - Sair"""


def menu():
    colonia = criar_colonia()
    while True:
        exibir(MENU)
        escolha = pedir("Opção: ").strip()
        if escolha == "1":
            mostrar_painel(colonia)
        elif escolha == "2":
            opcao_navegar(colonia)
        elif escolha == "3":
            opcao_analisar()
        elif escolha == "4":
            opcao_prever()
        elif escolha == "5":
            opcao_decidir()
        elif escolha == "6":
            opcao_prioridades(colonia)
        elif escolha == "7":
            opcao_modo(colonia)
        elif escolha == "8":
            opcao_ciclo(colonia)
        elif escolha == "9":
            opcao_alterar(colonia)
        elif escolha == "0":
            exibir("Encerrando o sistema. Até logo!")
            return
        else:
            exibir("Opção inválida.")


def main():
    try:
        menu()
    except (KeyboardInterrupt, EOFError):
        exibir("\nSistema encerrado.")


if __name__ == "__main__":
    main()
